# Circuit Optimization & Transpilation


In [ ]:
!pip install --upgrade qiskit qiskit_ibm_runtime networkx pylatexenc

## Review

### What have we learned so far?

- **Qubits**
    - State vectors, Bra-ket notation, Basis states

- **Measurement**

    - Probability amplitudes, Born rule

- **Multiple Qubits**

    - Tensor product, Product states, Entangled states

- **Operators**

    - Unitary matrices, Hermitian matrices

- **Circuits**


## Qubits

### States as Vectors

***Qubits*** are often represented with a column vector called a ***ket***.  

Each ket has an associated row vector called a ***bra***.

$$
\begin{bmatrix}\phantom{A}\!\!\! \alpha \\ \beta\end{bmatrix}
\qquad \text{and} \qquad
\begin{bmatrix} \,\alpha^* & \beta^* \, \end{bmatrix}
$$

$\alpha$ and $\beta$ are complex numbers, and $\alpha^*$ and $\beta^*$ are their ***complex conjugates***.

Complex conjugation means that if $z = x + iy$, then $z^* = x - iy$.

Importantly, the state represented by $\alpha$ and $\beta$ is normalized:
$$
\vert \alpha \vert^2 + \vert \beta \vert ^2 = 1
$$

> Notes on terminology:  
> The term for converting a ket to a bra, or a bra to a ket, is taking the ***conjugate transpose***.    
> A quantum state follows the ***Euclidean*** norm.

### Vectors Need a Basis

We always choose a basis to work in.  The ***standard*** or ***computational basis*** states
in quantum computing are:
$$
\vert 0 \rangle = \begin{bmatrix} 1 \\ 0 \end{bmatrix}   \qquad
\text{and}
\qquad \vert 1 \rangle = \begin{bmatrix} 0 \\ 1 \end{bmatrix}
$$

A qubit in a ***pure state*** can be written in several equivalent ways:

$$
\vert \psi \rangle
\ \ =\ \
\alpha \vert 0 \!\: \rangle + \beta \vert 1\!\: \rangle
\ \ =\ \
\alpha \begin{bmatrix}1\\0\end{bmatrix} + \beta \begin{bmatrix}0\\1\end{bmatrix}
\ \ = \ \
\left[\begin{matrix}
\phantom{A}\!\!\!\alpha \\ \beta
\end{matrix}\right]
$$

This linear combination of basis states is called superposition.

> If a state can be described by a single state vector it is a pure state.
>
> The counterpart to pure states is the ***mixed state***, which will be introduced with noise on day five.


## Measurement

### Coefficients are Probability Amplitudes

By the ***Born rule***, probabilities are the squared magnitudes of the amplitudes.

The ***probability amplitudes*** for $\vert 0 \rangle$ and $\vert 1 \rangle$ are the coefficients $\alpha$ and $\beta$, so:
$$
P(0) = \vert \alpha \vert ^2 \qquad \text{and} \qquad P(1) = \vert \beta \vert^2
$$

This is why the coefficients must be normalized.
$$
\lvert \alpha \rvert ^2 + \lvert \beta \rvert^2 = 1
$$

Note, if $\alpha = x + iy$ then:
$$
\vert \alpha \vert ^2
\ = \
\alpha^* \alpha
\ = \
(x-iy)(x+iy)
\ =\
x^2 + y^2
$$

> The probabilities of all possible outcomes must sum to one.   
> This is the why quantum states are required to be normalized.


In [ ]:
from qiskit.quantum_info import Statevector

psi = Statevector([0.6, 0.8])
p = psi.probabilities_dict()

print('P(0) =', p['0'])
print('P(1) =', p['1'])
print()
print('P(0) + P(1) =', p['0'] + p['1'])

### Finding an Outcome

Measuring in the computational basis gives either 0 or 1.
$$
\vert \psi \rangle
= \frac{\sqrt 3}{2} \vert 0 \rangle +
\frac1 2 \vert 1 \rangle
$$

Applying the Born rule to $\vert \psi \rangle$, the probabilities of measuring either 0 or 1 are:
$$
P(0) = \left \vert \frac{\sqrt 3} 2 \right \vert ^2 = \ \frac 3 4
\qquad \qquad
P(1) = \left\vert \frac{1}{2} \right\vert^2 = \  \frac 1 4
$$

After measurement, the qubit's state becomes the basis state of the measured outcome.
$$
0 \rightarrow \vert 0\rangle \qquad \qquad 1 \rightarrow \vert 1 \rangle
$$

> For any single measurement, only one outcome will be produced.  Which outcome occurs is random, based on the probabilities given by Born rule.
>
> The measured outcome becomes the new state.  If immediately measured again in the same basis, the same outcome will be obtained.
>
> This change from an arbitrary superposition to a single basis state is often referred to as ***collapse***.

In [ ]:
for k, v in psi.sample_counts(1000).items():
    print(v, 'of 1000 measurements had outcome', k)

## Multiple Qubits
### Two Qubits, Four Basis States

Two qubits have four associated basis states:
$$
\vert 00 \rangle, \  \vert 01 \rangle, \  \vert 10\rangle, \  \vert 11 \rangle
$$

The associated state vector has four terms:
$$
\vert \psi \rangle
=
a\vert 00 \rangle
+
b \vert 01 \rangle
+
c \vert 10 \rangle
+
d \vert 11 \rangle
$$

Amplitudes determine the probabilities of each result, so they must be normalized:
$$\vert a \vert ^2
+
\vert b \vert ^2
+
\vert c \vert ^2
+
\vert d \vert ^2
=
1
$$

> Generally, with $N$ qubits there are $2^N$ computational basis states, each with its own amplitude.



### Combining States

The ***tensor product*** (also known as the ***Kronecker product***) combines the states of multiple qubits.
$$
\vert 0 \rangle
\otimes
\vert 1 \rangle
=
\vert 01 \rangle
\qquad \qquad
\vert 1 \rangle
\otimes
\vert 0 \rangle
=
\vert 10 \rangle
$$


Generally:
$$
\begin{bmatrix}
a \\ b
\end{bmatrix}
\otimes
\begin{bmatrix}
c \\ d
\end{bmatrix}
=
\begin{bmatrix}
ac \\
ad \\
bc \\
bd
\end{bmatrix}
$$

> To save space in kets the tensor product symbol is almost always dropped: $\vert 0 \rangle \otimes \vert 1 \rangle = \vert 01 \rangle$
>
> Each additional qubit doubles the size of the resulting state vector.

In [ ]:
sv0 = Statevector.from_label('0')
sv1 = Statevector.from_label('1')

sv01 = sv0.tensor(sv1)

print(sv01 == Statevector.from_label('01'))

### States We Can Separate

If a multi-qubit state can be written as a tensor product of individual pure states it is called a ***product state***.
$$
\vert \!\:{+}\!\: \rangle = \frac{\vert 0\rangle+\vert 1 \rangle}{\sqrt 2}
$$
$$
\vert \!\:{+}\!\: \rangle
\otimes
\vert \!\:{+}\!\: \rangle
= \frac{1}{2} \Big(
\vert 00 \rangle +
\vert 01 \rangle +
\vert 10 \rangle +
\vert 11 \rangle
\Big)
$$

However, some states cannot be written as a product state, for example:
$$\vert \Phi^+ \rangle = \frac{\vert 00 \rangle + \vert 11 \rangle}{\sqrt 2}$$

In [ ]:
plus = Statevector.from_label('+')

plus.tensor(plus).draw('latex')

### States We Cannot Separate

When a state cannot be written as a tensor product of individual pure states it is
called an ***entangled state***.
$$
\vert \Phi^+ \rangle = \frac{\vert 00 \rangle + \vert 11 \rangle}{\sqrt 2}
$$

A measurement of either qubit determines the outcome of a measurement of the other qubit.

For $\vert \Phi^+ \rangle$, the only possible outcomes for a measurement in the computational
basis are $\vert 00 \rangle$ and $\vert 11 \rangle$, each with probability $1/2$.

> This is one of four ***Bell states**, a standard set of maximally entangled two-qubit states.


In [ ]:
from math import sqrt

bell = Statevector([1/sqrt(2), 0, 0, 1/sqrt(2)])

for k, v in bell.probabilities_dict().items():
    print(f'P({k}) = {v}')

## Operators

### Matrices Act on States

***Operators*** are linear transformations that act on the vectors representing quantum states.  They are represented by matrices:
$$
X = \begin{bmatrix}
0 & 1 \\
1 & 0
\end{bmatrix}
$$

If an operator $A$ acts on $\vert \psi \rangle$, then:  $\ \ A \vert \psi \rangle = \vert \psi' \rangle$
$$
X \vert 0 \rangle =
\begin{bmatrix}
0 & 1 \\
1 & 0
\end{bmatrix}
\begin{bmatrix}
1 \\ 0
\end{bmatrix} =
\begin{bmatrix}
0 \\ 1
\end{bmatrix}
$$

$$
X \vert 1 \rangle =
\begin{bmatrix}
0 & 1 \\
1 & 0
\end{bmatrix}
\begin{bmatrix}
0 \\ 1
\end{bmatrix} =
\begin{bmatrix}
1 \\ 0
\end{bmatrix}
$$

> This operator $X$ is known as the ***bit-flip*** operator.
>
> Not every matrix represents a physically allowed operation.  We will talk about the required properties soon.


In [ ]:
from qiskit.quantum_info import Operator

X = Operator.from_label('X')
I = X.dot(X)

sv0 = Statevector.from_label('0')
sv1 = Statevector.from_label('1')

print('X|0> = |1> ? ', sv0.evolve(X) == sv1)
print('X|1> = |0> ? ', sv1.evolve(X) == sv0)

### Evolution and Measurement

**Unitary operators** describe reversible evolutions of quantum states.  Matrices are unitary when:
$$
U^\dagger U = UU^\dagger = I
$$

***Hermitian operators*** describe ***observables***, their real eigenvalues are the possible measurement outcomes for that observable.
$$
A^\dagger = A
$$


> The symbol $\dagger$, read as “dagger,” indicates the conjugate transpose operation, also known as the adjoint.
>
> Multiplication by a unitary matrix preserves normalization when acting on a state vector.  Later you will see how this preserves probabilities.
>
> Hermitian operators are their own conjugate transpose. Because of this, their eigenvalues are real numbers.  Since those are the possible measurement outcomes of the operator, they must be real valued.

## Circuits

### Wires and Gates

It is common to represent qubits and operators as a circuit.

Qubits are represented as wires, with each gate acting on them from the left to
the right.

Single qubit gates:   
$$ X, Y, Z, H $$

Multi-qubit gates:
$$ CX, \ SW\!AP $$

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)

qc.h(0)
qc.cx(0, 1)
qc.x(1)

qc.measure_all()

qc.draw('mpl')

## Topics

- **Circuit Depth**
    - Gates Take Time, Why Depth Matters, Same Result Different Depth
- **Reducing Depth**
    - The GHZ State, Fan-Out vs. Chain, Recursive Fan-Out
- **Transpilation**
    - Hardware Has Rules, The Coupling Map, What the Transpiler Does, Transpilation Can Increase Depth
- **Heavy-Hex Topology**
    - How Heron Qubits Connect, GHZ on Heavy-Hex



## Circuit Depth

### Gates Take Time

Gates represent operations being done on qubits, but they must be done in the
correct sequence and each operation takes time to complete.

Operations on different qubits can often happen at the same time.

Operations that depend on each other or act on the same qubit must happen sequentially.

### Why Depth Matters

So far we have discussed **abstract circuits**.  These are idealized with perfect qubits,
perfect operators, and complete connectivity between qubits.

A qubit in a real quantum computer is a physical object.

The qubit's environment can disturb it, the equipment applying operators to it is
imperfect, and normally not all qubits in the quantum computer can communicate
directly with each other.

A circuit's ***depth*** is the maximum number of sequential operations it requires.

"Shallow" circuits, with less depth, generally have less opportunity for these errors to
accumulate.

> Noise from the environment and gate errors will be discussed in tomorrow's session.
>
> Connectivity between qubits is discussed later in this session.


In [ ]:
print(qc.depth())

### Same Result, Different Depth

Both circuits result in the same state:

$$\vert \Phi^+ \rangle = \frac{\vert 00 \rangle + \vert 11 \rangle}{\sqrt 2}$$


In [ ]:
qc2 = QuantumCircuit(2)

qc2.h(0)
qc2.x(1)
qc2.cx(0,1)

qc2.draw('mpl')

In [ ]:
qc3 = QuantumCircuit(2)

qc3.h(0)
qc3.cx(0,1)
qc3.x(1)

qc3.draw('mpl')

In [ ]:
print('Depth of qc2:', qc2.depth())
print('Depth of qc3:', qc3.depth())
print()
print('qc2 == qc3? ', Statevector.from_circuit(qc2) == Statevector.from_circuit(qc3))

## Reducing Depth

### The GHZ State

This is the ***fan-out*** method to prepare an $N$-bit GHZ state:
- apply $H$ to $q_0$,
- followed by a $CX$ gate from $q_0$ to each of the other qubits.

Each $CX$ gate acts on $q_0$, so they must be applied sequentially.

For an $n$-qubit GHZ state the circuit will be depth $1 + (n -1) = n$.

In [ ]:
n = 4

qc_fan = QuantumCircuit(n)

qc_fan.h(0)

for i in range(1, n):
    qc_fan.cx(0, i)

qc_fan.draw('mpl')

In [ ]:
print(f'Depth of {n}-qubit fan-out GHZ state:  {qc_fan.depth()}')

### Fan-Out vs. Chain

This is the ***chain*** method of creating a GHZ state.

Each $CX$ gate has a different control qubit, but they still act on qubits used by a previous gate.

Depth $= 1 + (n −1) = n$

Once two qubits are entangled, either qubit can act as the control qubit of a later gate.



In [ ]:
qc_chain = QuantumCircuit(n)

qc_chain.h(0)

for i in range(n-1):
    qc_chain.cx(i, i+1)

qc_chain.draw('mpl')

In [ ]:
print(f'Depth of {n}-qubit chain GHZ state:  {qc_chain.depth()}')

### Recursive Fan-Out

Recursive fan-out uses all entangled qubits as control qubits in every layer.

Depth $= 1 + \lceil\!\: \log_2(n) \rceil$

> Recursive fanout doubles the number of entangled qubits for each added layer of depth.
>
> That notation is the ***ceiling*** function.  It forces floating point numbers up to the next largest integer.
>
> Note that the diagroms for the circuits below might make them look different depths, but if you examine the first diagram closely you will see that the second and third $CX$ gates do not use the same qubits.  That means they can run simultaneously.


In [ ]:
qc_rec1 = QuantumCircuit(n)

qc_rec1.h(0)

for i in range(1, n):
    c = (i - 1) // 2
    qc_rec1.cx(c, i)

qc_rec1.draw('mpl')

In [ ]:
qc_rec2 = QuantumCircuit(n)

qc_rec2.h(0)

l = 2**(n.bit_length())
while l > 1:
    for i in range(0, n, l):
        if i+(l//2) < n:
            qc_rec2.cx(i, i+(l//2))
    l //= 2

qc_rec2.draw('mpl')

In [ ]:
print(f'Depth of {n}-qubit recursive method 1 GHZ state:  {qc_rec1.depth()}')
print(f'Depth of {n}-qubit recursive method 2 GHZ state:  {qc_rec2.depth()}')

## Transpilation

### Hardware Has Rules

So far we have optimized an abstract circuit for the GHZ state.

Real hardware imposes constraints on which qubits can directly interact and
what gates are natively supported.

***Transpilation*** is the conversion of an abstract circuit into one that can be run on
a specific quantum computer.

The result of transpilation is often called an ***ISA circuit***, referring to the target
quantum computer's ***Instruction Set Architecture***.



### The Coupling Map

Nodes on the coupling map are physical qubits, and the edges
are the physical couplers between them.

Numbers on the nodes are the physical qubit indices, and may
not match the abstract circuit's qubit indices.

Two-qubits gates (like $CX$) cannot be made between physical
qubits without coupling map connections.

If two logical qubits need to interact the transpiler must either
map them to physically connected qubits in the initial layout or
use $SW\!AP$ gates during routing to move their states together.

> There is a `CouplingMap.draw()` function, but it requires installing the GraphViz utility.
>
> The code below draws the coupling map without needing to install GraphViz.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from qiskit_ibm_runtime.fake_provider import FakeFez

G = nx.Graph()
G.add_edges_from(FakeFez().coupling_map.get_edges())

plt.figure(figsize=(20, 16))

pos = nx.spring_layout(G, seed=1234, k=6, iterations=10000)

nx.draw(
    G, pos, with_labels=True, node_color="lightblue",
    node_size=1000, edgecolors="black", edge_color="gray",
    width=2, font_size=12, font_color="black", font_weight="bold"
)

plt.show()

### What the Transpiler Does

The Qiskit transpiler by default has six stages:
$$
\text{Init} \rightarrow
\text{Layout} \rightarrow
\text{Routing} \rightarrow
\text{Translation} \rightarrow
\text{Optimization} \rightarrow
\text{Scheduling}
$$

***Layout*** maps the abstract circuit's qubits to physical qubits.

***Routing*** identifies where $SW\!AP$ gates need to be inserted to move states to qubits that are
physically connected, so that two-qubit gates can be run on them.

***Translation*** converts the abstract circuit's gates to the native gate set of the target
quantum computer.

***Optimization*** attempts to simplify the circuit.

> ***Init*** and ***Scheduling*** are normally placeholders that do no work unless you have specifically added init or scheduling passes to the pass manager.
>
> Remember $SW\!AP$ gates do not "move qubits".  The qubits are physical systems, the $SW\!AP$ gates exchange the quantum state of the qubits.
>
> Routing and translation tend to add operations.  Optimization tests to remove or simplify them.
>
> The final ISA circuit is generally deeper than the original abstract circuit.


In [ ]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(
    optimization_level=0,
    backend=FakeFez()
)

### Transpilation Can Increase Depth

***Universal gate sets*** can be used to implement any quantum computation.

Quantum processors natively implement a small ***basis gate set***.  All other gates
must be translated into combinations of those gates.

Heron QPU basis gates: $CZ$, $RX$, $RZ$, $SX$, $X$

> There are multiple universal gate sets.  Generally, QPU basis gate sets are universal.
>
> The translation of abstract circuit gates to the basis gate set is on reason transpilation tends to increase circuit depth.



In [ ]:
qc_abs = QuantumCircuit(2)

qc_abs.h(0)
qc_abs.cx(0, 1)

qc_abs.draw('mpl')

In [ ]:
qc_isa = pm.run(qc_abs)

qc_isa.draw('mpl')

In [ ]:
print('Abstract circuit depth:   ', qc_abs.depth())
print('Transpiled circuit depth: ', qc_isa.depth())

## Heavy-Hex Topology

### How Heron Qubits Connect

Heavy-hex is the topology used on Heron QPUs.

Most qubits are connected directly to 2 or 3
others.

This simplifies hardware engineering, but restricts
circuit layout.

This means that good layout and routing during
transpilation is very important.

> The topology of a QPU's coupling map is an engineering tradeoff.
>
> IBM's new Nighthawk QPUs use a more densely connected square lattice, but Heron is the generation of QPU available to users on the free Open plan.
>
> The image in the slideshow at this point is taken from:  
> [https://quantum.cloud.ibm.com/computers?system=ibm_fez](https://quantum.cloud.ibm.com/computers?system=ibm_fez)

### GHZ on Heavy-Hex

The abstract recursive fan-out GHZ circuit can
entangle 256 qubits with 8 layers of $CX$
gates.


Starting from physical qubit 64 on this Heavy-Hex coupling map we are only able to entangle
42 qubits with 8 layers of $CX$
gates.


Hardware connectivity limits how quickly
entanglement can be spread.

> The results for this do depend on starting qubit.
>
> As entanglement spreads the degree-3 qubits will have at most two unentangled neighbors to spread to, but the degree-2 qubits will only have one neighbor to spread to.


In [ ]:
CENTER_NODE = 88
NUM_LAYERS = 8
LAYER_COLORS = [
    "#FFF176",  # yellow
    "#FFD54F",  # golden yellow
    "#FFB74D",  # light orange
    "#FF9800",  # orange
    "#F57C00",  # dark orange
    "#EF5350",  # light red
    "#D32F2F",  # red
    "#B71C1C",  # dark red
]

DEFAULT_COLOR = "white"

G = nx.Graph()
G.add_edges_from(FakeFez().coupling_map.get_edges())

def recursive_fanout_layers(G, start, num_layers):
    """
    Returns:
        layers:
            layers[0] = [start]
            layers[k] = qubits newly added during CX layer k

        cx_layers:
            cx_layers[k-1] = [(control, target), ...]
            CX gates that can execute in parallel in that layer.
    """

    entangled = {start}

    layers = [[start]]
    cx_layers = []

    for _ in range(num_layers):
        used_this_layer = set()
        new_qubits = []
        cx_gates = []

        # Only qubits entangled BEFORE this layer may act as controls.
        controls = list(entangled)

        for control in controls:
            if control in used_this_layer:
                continue

            # Find an unused neighbor not already in the GHZ state.
            candidates = [
                neighbor
                for neighbor in G.neighbors(control)
                if neighbor not in entangled
                and neighbor not in used_this_layer
            ]

            if not candidates:
                continue

            # Deterministic choice.
            # Change this policy if you want different growth behavior.
            target = min(candidates)

            cx_gates.append((control, target))
            new_qubits.append(target)

            used_this_layer.add(control)
            used_this_layer.add(target)

        if not new_qubits:
            break

        entangled.update(new_qubits)

        layers.append(new_qubits)
        cx_layers.append(cx_gates)

    print(f'Only {len(entangled)} qubits in GHZ')
    return layers, cx_layers


layers, cx_layers = recursive_fanout_layers(
    G,
    CENTER_NODE,
    NUM_LAYERS,
)

node_layer = {}

for layer_index, nodes in enumerate(layers):
    for node in nodes:
        node_layer[node] = layer_index


def get_node_color(node):
    layer = node_layer.get(node)

    if layer is None:
        return DEFAULT_COLOR

    if layer < len(LAYER_COLORS):
        return LAYER_COLORS[layer]

    return DEFAULT_COLOR


node_colors = [
    get_node_color(node)
    for node in G.nodes
]

In [ ]:
plt.figure(figsize=(20, 16))

pos = nx.spring_layout(G, seed=1234, k=6, iterations=10000)

nx.draw_networkx_edges(G, pos, edge_color="gray", width=1.5,)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, edgecolors="black", linewidths=1.2, node_size=500)

nx.draw_networkx_labels(G, pos, font_size=8)

plt.axis("off")
plt.tight_layout()
plt.show()

## Questions?

